In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, regexp_replace, col, lit, expr
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

############################
#####--TRANFORMATIONS--#####
############################

def transform_customers(df):
    #split_col = split(df['customer_name'],',')
    df = df.withColumn('name_parts',split(col('customer_name'),','))\
            .withColumn('first_name',expr('get(name_parts, 1)'))\
            .withColumn('last_name',expr('get(name_parts, 0)'))\
            .drop('name_parts')\
        .withColumn('postcode',regexp_replace(col('postcode'), r'\.0$', ''))\
        .withColumn('country', lit("USA"))\
        .drop('customer_name','file_path')

    # Handling duplicates rows
    df = df.dropDuplicates(['customer_id']) 

    df = df.select(
    'customer_id',
    'first_name',
    'last_name',
    'tax_id',
    'tax_code',
    'state',
    'city',
    'postcode',
    'street',
    'number',
    'unit',
    'region',
    'district',
    'country',
    'lon',
    'lat',
    'ship_to_address',
    'valid_from',
    'valid_to',
    'units_purchased',
    'loyalty_segment',
    'last_update_ts'
    )

    return df

##################################
#####--SCD-1 IMPLEMENTATION--#####
##################################

def scd_merge_table(spark, source_table, target_table, business_key):

    # source_df = spark.table(source_table)

    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table")
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print("First Load: Silver Table Created")

    else:
        print("Incremental Load: Performing SCD type 1 Merge") 
        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = "AND".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )
        ## ouptput of above merge_condition:
        ## e.g. ["target.customer_id = source.customer_id" and "target.product_id = source.product_id" etc.]

        delta_table.alias("target").merge(source_table.alias("source"),merge_condition,)\
                                        .whenMatchedUpdateAll()\
                                        .whenNotMatchedInsertAll()\
                                        .execute()
        print("Incremental Load: SCD type 1 Merge Complete")

########################
#####--MAIN LOGIC--#####
########################

source_table = "ecommerce_analytics.bronze.customers"
target_table = "ecommerce_analytics.silver.silver_customers"
business_key = ['customer_id']
df = spark.read.table(source_table)

source_table_df = transform_customers(df)

scd_merge_table(spark, source_table_df, target_table, business_key)

display(spark.table(target_table))

# Process

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import split

split_col = split(df['customer_name'],',')

df = df.withColumn('first_name',split_col.getItem(1))\
    .withColumn('last_name',split_col.getItem(0))\
    .drop('customer_name')

df.display()


In [0]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn('postcode',regexp_replace(col('postcode'), r'\.0$', ''))

df.display()

In [0]:
df.columns

In [0]:
# check if silver table exists already before perforing any transformations

spark.catalog.tableExists("ecommerce_analytics.silver.customers")

In [0]:
# SCD-1 
# - Update/Replace changed data
# - Add new data
from delta.tables import DeltaTable

if not spark.catalog.tableExists("ecommerce_analytics.silver.customers"):
    print("First Load: Creating Silver Table")
    df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_analytics.silver.customers")
else:
    print("Incremental Load: Performing SCD type 1 Merge") 
    delta_table = DeltaTable.forName(spark, "ecommerce_analytics.silver.customers")

    merge_condition = "target.customer_id = source.customer_id"

    delta_table.alias("target").merge(
        df.alias("source"),
        merge_condition,
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()